In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================================
# PARAMETERS
# ==========================================================

DB = "customer_health"
BRONZE = "BRONZE"
SILVER = "SILVER"
GOLD = "GOLD"

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

try:

    # ==========================================================
    # CREATE TARGET TABLES IF NOT EXISTS
    # ==========================================================

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB}.{GOLD}.LEAD_BASIC_DETAILS
    (
        lead_id STRING,
        lead_route STRING,
        lead_source STRING,
        funnel_id STRING,
        lead_tag STRING,
        funnel STRING,
        status_label STRING,
        avatar STRING,
        lead_quality STRING,
        current_salary STRING,
        learning_before STRING,
        career_goal_timeline STRING,
        us_citizen STRING,
        credit_score STRING,
        date_created TIMESTAMP,
        insert_date TIMESTAMP,
        final_funnel STRING
    )
    USING DELTA
    """)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB}.{SILVER}.LEADS_PROCESSED
    (
        display_name STRING,
        contact_id STRING,
        lead_id STRING,
        name STRING,
        email STRING,
        lead_owner STRING,
        phone STRING,
        md5_hash STRING,
        insert_date TIMESTAMP,
        update_date TIMESTAMP
    )
    USING DELTA
    """)

    # ==========================================================
    # READ ALL RAW DATA (parse once, filter per target)
    # ==========================================================

    raw_df = spark.table(f"{DB}.{BRONZE}.LEADS_RAW")

    # Parse JSON once for reuse
    parsed_df = (
        raw_df
        .withColumn(
            "json_string",
            F.expr(
                "customer_health.silver.parse_activity_final(raw_data)"
            )
        )
    )

    # ==========================================================
    # LEAD BASIC DETAILS - INCREMENTAL LOAD
    # ==========================================================

    max_insert_date_lead = spark.sql(f"""
        SELECT COALESCE(
            MAX(insert_date),
            TIMESTAMP('1900-01-01')
        ) AS max_dt
        FROM {DB}.{GOLD}.LEAD_BASIC_DETAILS
    """).first()["max_dt"]

    # Filter for new records
    lead_parsed_df = (
        parsed_df
        .filter(
            F.col("insert_date") > F.lit(max_insert_date_lead)
        )
    )

    if lead_parsed_df.limit(1).count() > 0:

        print(f"Processing {lead_parsed_df.count()} new records for LEAD_BASIC_DETAILS")

        lead_basic_df = (
            lead_parsed_df
            .select(
                F.get_json_object(
                    "json_string",
                    "$.id"
                ).alias("lead_id"),

            F.coalesce(
                F.get_json_object(
                    "json_string",
                    "$.custom.lead_route"
                ),
                F.lit("NO_ROUTE")
            ).alias("lead_route"),

            F.get_json_object(
                "json_string",
                "$.custom.Lead Source"
            ).alias("lead_source"),

            F.get_json_object(
                "json_string",
                "$.custom.Funnel ID"
            ).alias("funnel_id"),

            F.get_json_object(
                "json_string",
                "$.custom.tag"
            ).alias("lead_tag"),

            F.get_json_object(
                "json_string",
                "$.custom.Funnel"
            ).alias("funnel"),

            F.get_json_object(
                "json_string",
                "$.status_label"
            ).alias("status_label"),

            F.get_json_object(
                "json_string",
                "$.custom.Avatar"
            ).alias("avatar"),

            F.get_json_object(
                "json_string",
                "$.custom.Lead Quality"
            ).alias("lead_quality"),

            F.get_json_object(
                "json_string",
                "$.custom.Current Salary"
            ).alias("current_salary"),

            F.get_json_object(
                "json_string",
                "$.custom.Learning Before"
            ).alias("learning_before"),

            F.get_json_object(
                "json_string",
                "$.custom.career goal timeline"
            ).alias("career_goal_timeline"),

            F.get_json_object(
                "json_string",
                "$.custom.Visa Status"
            ).alias("us_citizen"),

            F.get_json_object(
                "json_string",
                "$.custom.[PF] Estimated Credit Score"
            ).alias("credit_score"),

            F.to_timestamp(
                F.get_json_object(
                    "json_string",
                    "$.date_created"
                )
            ).alias("date_created"),

            F.col("insert_date")
        )
        )

        lead_basic_df = (
            lead_basic_df
            .withColumn(
                "final_funnel",
                F.coalesce(
                    F.when(
                        F.col("funnel").isNotNull(),
                        F.col("funnel")
                    )
                    .when(
                        (F.col("funnel_id").isNull()) &
                        (
                            F.lower(
                                F.col("lead_tag")
                            ).contains("ulinc")
                        ),
                        F.lit("unlinc")
                    )
                    .when(
                        (F.col("funnel").isNull()) &
                        (F.col("funnel_id").isNull()) &
                        (F.col("lead_tag").isNull()) &
                        (
                            F.col("lead_source")
                            .like("%Stripe%")
                        ),
                        F.lit("website_signup")
                    )
                    .otherwise(
                        F.col("funnel_id")
                    ),
                    F.lit("NO_FUNNEL")
                )
            )
        )

        window_spec = Window.partitionBy(
            "lead_id"
        ).orderBy(
            F.col("insert_date").desc()
        )

        lead_basic_df = (
            lead_basic_df
            .withColumn(
                "rn",
                F.row_number().over(
                    window_spec
                )
            )
            .filter(
                F.col("rn") == 1
            )
            .drop("rn")
        )

        # ==========================================================
        # MERGE LEAD_BASIC_DETAILS
        # ==========================================================

        lead_target = DeltaTable.forName(
            spark,
            f"{DB}.{GOLD}.LEAD_BASIC_DETAILS"
        )

        (
            lead_target.alias("tgt")
            .merge(
                lead_basic_df.alias("src"),
                "tgt.lead_id = src.lead_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("✅ LEAD_BASIC_DETAILS updated")

    else:
        print("No new records for LEAD_BASIC_DETAILS")

    # ==========================================================
    # LEADS_PROCESSED - INCREMENTAL LOAD
    # ==========================================================

    max_insert_date_processed = spark.sql(f"""
        SELECT COALESCE(
            MAX(insert_date),
            TIMESTAMP('1900-01-01')
        ) AS max_dt
        FROM {DB}.{SILVER}.LEADS_PROCESSED
    """).first()["max_dt"]

    # Filter for new records
    processed_parsed_df = (
        parsed_df
        .filter(
            F.col("insert_date") > F.lit(max_insert_date_processed)
        )
    )

    if processed_parsed_df.limit(1).count() > 0:

        print(f"Processing {processed_parsed_df.count()} new records for LEADS_PROCESSED")

        contacts_df = (
            processed_parsed_df
            .withColumn(
            "contact",
            F.explode(
                F.from_json(
                    F.get_json_object(
                        "json_string",
                        "$.contacts"
                    ),
                    """
                    array<
                        struct<
                            id:string,
                            lead_id:string,
                            name:string,
                            display_name:string,
                            emails:array<
                                struct<
                                    email:string
                                >
                            >,
                            phones:array<
                                struct<
                                    phone:string
                                >
                            >
                        >
                    >
                    """
                )
            )
        )
        .withColumn(
            "email",
            F.explode_outer(
                "contact.emails"
            )
            )
        )

        leads_processed_df = (
            contacts_df
            .select(
                F.col(
                    "contact.display_name"
                ).alias("display_name"),

                F.col(
                    "contact.id"
                ).alias("contact_id"),

                F.col(
                    "contact.lead_id"
                ).alias("lead_id"),

                F.col(
                    "contact.name"
                ).alias("name"),

                F.col(
                    "email.email"
                ).alias("email"),

                F.get_json_object(
                    "json_string",
                    "$.custom.Lead Owner"
                ).alias("lead_owner"),

                F.expr(
                    "get(contact.phones, 0).phone"
                ).alias("phone")
            )
            .distinct()
        )

        leads_processed_df = (
            leads_processed_df
            .withColumn(
                "md5_hash",
                F.md5(
                    F.concat_ws(
                        "||",
                        *[
                            F.coalesce(
                                F.col(c),
                                F.lit("")
                            )
                            for c in [
                                "display_name",
                                "contact_id",
                                "lead_id",
                                "name",
                                "email",
                                "lead_owner",
                                "phone"
                            ]
                        ]
                    )
                )
            )
        )

        # ==========================================================
        # MERGE LEADS_PROCESSED
        # ==========================================================

        processed_target = DeltaTable.forName(
            spark,
            f"{DB}.{SILVER}.LEADS_PROCESSED"
        )

        (
            processed_target.alias("tgt")
            .merge(
                leads_processed_df.alias("src"),
                """
                tgt.lead_id = src.lead_id
                AND tgt.email = src.email
                AND tgt.contact_id = src.contact_id
                """
            )
            .whenMatchedUpdate(
                condition="""
                tgt.md5_hash <> src.md5_hash
                """,
                set={
                    "display_name":
                        "src.display_name",
                    "name":
                        "src.name",
                    "lead_owner":
                        "src.lead_owner",
                    "phone":
                        "src.phone",
                    "md5_hash":
                        "src.md5_hash",
                    "update_date":
                        "current_timestamp()"
                }
            )
            .whenNotMatchedInsert(
                values={
                    "display_name":
                        "src.display_name",
                    "contact_id":
                        "src.contact_id",
                    "lead_id":
                        "src.lead_id",
                    "name":
                        "src.name",
                    "email":
                        "src.email",
                    "lead_owner":
                        "src.lead_owner",
                    "phone":
                        "src.phone",
                    "md5_hash":
                        "src.md5_hash",
                    "insert_date":
                        "current_timestamp()",
                    "update_date":
                        "current_timestamp()"
                }
            )
            .execute()
        )

        print("✅ LEADS_PROCESSED updated")

    else:
        print("No new records for LEADS_PROCESSED")

    print(
        "\n✅ Pipeline Executed Successfully"
    )

except Exception as e:

    raise Exception(
        f"Error: {str(e)}"
    )